# Setup

In [5]:
!./.venv/bin/pip install pandas scikit-learn numpy kaggle xgboost matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 32.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 74.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]


In [3]:
!./.venv/bin/pip install "dask[complete]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.2/34.2 MB 44.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 73.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 73.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [distributed] [distributed]


In [20]:
!./venv/bin/pip install "pyarrow>=13.0.0"


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [21]:
!./venv/bin/pip install --upgrade --force-reinstall \
    pandas \
    pyarrow \
    'snowflake-connector-python[pandas]' \
    sqlalchemy \
    snowflake-sqlalchemy

  Using cached pandas-3.0.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached pyarrow-23.0.1-cp314-cp314-macosx_12_0_arm64.whl.metadata (3.1 kB)
  Using cached sqlalchemy-2.0.48-cp314-cp314-macosx_11_0_arm64.whl.metadata (9.5 kB)
  Using cached snowflake_sqlalchemy-1.9.0-py3-none-any.whl.metadata (37 kB)
  Using cached snowflake_connector_python-4.3.0-cp314-cp314-macosx_13_0_arm64.whl
  Using cached numpy-2.4.3-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached asn1crypto-1.5.1-py2.py3-none-any.whl.metadata (13 kB)
  Using cached cryptography-46.0.5-cp311-abi3-macosx_10_9_universal2.whl.metadata (5.7 kB)
  Using cached pyopenssl-25.3.0-py3-none-any.whl.metadata (17 kB)
  Using cached pyjwt-2.12.1-py3-none-any.whl.metadata (4.1 kB)
  Using cached pytz-2026.1.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using 

In [ ]:
!kaggle datasets download -d thegurusteam/spanish-high-speed-rail-system-ticket-pricing --unzip

Dataset URL: https://www.kaggle.com/datasets/thegurusteam/spanish-high-speed-rail-system-ticket-pricing
License(s): GPL-2.0
100%|████████████████████████████████████████| 557M/557M [00:30<00:00, 18.9MB/s]



# Imports

In [ ]:
import pandas as pd

In [26]:
ddf = pd.read_csv("renfe_min.csv")

In [27]:
ddf.isna().sum()

id                  0
company             0
origin              0
destination         0
departure           0
arrival             0
duration            0
vehicle_type        0
vehicle_class       2
price             222
fare                2
seats            2000
meta                0
insert_date         0
dtype: int64

In [22]:
# Parse datetime columns and split into components
ddf["departure"] = pd.to_datetime(ddf["departure"], errors="coerce")
ddf["arrival"] = pd.to_datetime(ddf["arrival"], errors="coerce")

for col in ["departure", "arrival"]:
    ddf[f"{col}_month"] = ddf[col].dt.month.astype("Int64")
    ddf[f"{col}_day"] = ddf[col].dt.day.astype("Int64")
    ddf[f"{col}_hour"] = ddf[col].dt.hour.astype("Int64")
    ddf[f"{col}_minute"] = ddf[col].dt.minute.astype("Int64")

ddf[[
    "departure_month", "departure_day", "departure_hour", "departure_minute",
    "arrival_month", "arrival_day", "arrival_hour", "arrival_minute",
]].head()

,departure_month,departure_day,departure_hour,departure_minute,arrival_month,arrival_day,arrival_hour,arrival_minute
0,4,18,5,50,4,18,8,55
1,4,18,13,25,4,18,16,24
2,4,18,6,30,4,18,9,20
3,4,18,15,30,4,18,18,40
4,4,18,7,0,4,18,9,30


In [5]:
ddf.describe()

,id,duration,price,seats
count,2000.000000,2000.00000,1778.000000,0.0
mean,1000.500000,2.96215,64.709556,NaN
std,577.494589,1.41365,26.272846,NaN
min,1.000000,1.67000,15.450000,NaN
25%,500.750000,2.50000,45.800000,NaN
50%,1000.500000,2.61500,64.050000,NaN
75%,1500.250000,2.98000,76.300000,NaN
max,2000.000000,9.37000,214.200000,NaN


In [23]:
ddf.sample(5)

,id,company,origin,destination,departure,arrival,duration,vehicle_type,vehicle_class,price,...,meta,insert_date,departure_month,departure_day,departure_hour,departure_minute,arrival_month,arrival_day,arrival_hour,arrival_minute
1102,1103,renfe,MADRID,BARCELONA,2019-05-21 19:00:00,2019-05-21 21:30:00,2.50,AVE,Turista,78.80,...,{},2019-04-11 21:53:31,5,21,19,0,5,21,21,30
421,422,renfe,MADRID,SEVILLA,2019-05-06 14:00:00,2019-05-06 16:32:00,2.53,AVE,Turista,47.30,...,{},2019-04-11 21:50:52,5,6,14,0,5,6,16,32
1029,1030,renfe,MADRID,VALENCIA,2019-06-07 06:08:00,2019-06-07 12:55:00,6.78,REGIONAL,Turista,28.35,...,{},2019-04-11 21:53:12,6,7,6,8,6,7,12,55
457,458,renfe,MADRID,SEVILLA,2019-05-10 12:00:00,2019-05-10 14:32:00,2.53,AVE,Turista,60.30,...,{},2019-04-11 21:51:08,5,10,12,0,5,10,14,32
58,59,renfe,MADRID,BARCELONA,2019-05-18 15:30:00,2019-05-18 18:40:00,3.17,AVE,Preferente,86.80,...,{},2019-04-11 21:49:48,5,18,15,30,5,18,18,40


In [24]:
ddf.drop(columns=["seats", "meta", "insert_date", "id", "company"])

,origin,destination,departure,arrival,duration,vehicle_type,vehicle_class,price,fare,departure_month,departure_day,departure_hour,departure_minute,arrival_month,arrival_day,arrival_hour,arrival_minute
0,MADRID,BARCELONA,2019-04-18 05:50:00,2019-04-18 08:55:00,3.08,AVE,Preferente,68.95,Promo,4,18,5,50,4,18,8,55
1,MADRID,BARCELONA,2019-04-18 13:25:00,2019-04-18 16:24:00,2.98,AVE-TGV,Turista,107.70,Flexible,4,18,13,25,4,18,16,24
2,MADRID,BARCELONA,2019-04-18 06:30:00,2019-04-18 09:20:00,2.83,AVE,Turista,75.40,Promo,4,18,6,30,4,18,9,20
3,MADRID,BARCELONA,2019-04-18 15:30:00,2019-04-18 18:40:00,3.17,AVE,Preferente,NaN,Promo,4,18,15,30,4,18,18,40
4,MADRID,BARCELONA,2019-04-18 07:00:00,2019-04-18 09:30:00,2.50,AVE,Turista Plus,106.75,Promo,4,18,7,0,4,18,9,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,MADRID,BARCELONA,2019-04-30 10:30:00,2019-04-30 13:15:00,2.75,AVE,Turista,85.10,Promo,4,30,10,30,4,30,13,15
1996,MADRID,BARCELONA,2019-04-30 09:00:00,2019-04-30 11:45:00,2.75,AVE,Preferente,NaN,Promo,4,30,9,0,4,30,11,45
1997,MADRID,BARCELONA,2019-04-30 11:30:00,2019-04-30 14:40:00,3.17,AVE,Turista,85.10,Promo,4,30,11,30,4,30,14,40
1998,MADRID,BARCELONA,2019-04-30 19:30:00,2019-04-30 22:40:00,3.17,AVE,Turista,85.10,Promo,4,30,19,30,4,30,22,40


In [8]:
fare_counts = ddf["fare"].value_counts()

In [9]:
popular_fares = fare_counts[fare_counts > 50].index

In [10]:
ddf[ddf["fare"].isin(popular_fares)]["fare"].value_counts()

fare
Promo         1390
Flexible       474
Adulto ida     106
Name: count, dtype: int64

In [11]:
ddf[ddf["fare"].isin(popular_fares)]["fare"].unique()

array(['Promo', 'Flexible', 'Adulto ida'], dtype=object)

In [ ]:
ddf["vehicle_type"].value_counts()

vehicle_type
AVE          1476
ALVIA         150
REGIONAL       84
INTERCITY      82
AV City        74
MD-LD          36
AVE-TGV        30
AVE-MD         26
R. EXPRES      22
AVE-LD         20
Name: count, dtype: int64

In [30]:
ddf["vehicle_class"].value_counts()

vehicle_class
Turista               1510
Preferente             248
Turista Plus           174
Turista con enlace      66
Name: count, dtype: int64

In [35]:
def encode_vehicle_class_binary(series):
    # Preferente -> 0, all other classes -> 1
    s = series.fillna("").astype(str).str.upper().str.strip()
    return (~s.str.contains("PREFERENTE", regex=False)).astype("int8")


ddf["vehicle_class_bin"] = encode_vehicle_class_binary(ddf["vehicle_class"])
ddf[["vehicle_class", "vehicle_class_bin"]].head()

,vehicle_class,vehicle_class_bin
0,Preferente,0
1,Turista,1
2,Turista,1
3,Preferente,0
4,Turista Plus,1


In [ ]:
s = pd.Series(ddf["vehicle_type"].unique())

# normalize
s = s.str.upper().str.strip()

s = s.replace({
    "INTERCITY": "INTERCITY",
    "REG.EXP.": "REG.EXP.",
    "R. EXPRES": "REG.EXP.",  
})

# split to token lists
tokens = s.str.split("-")

# create multi-hot columns
X_vehicle = tokens.explode().str.get_dummies().groupby(level=0).max()

print(X_vehicle)

   ALVIA  AV CITY  AVE  INTERCITY  LD  MD  R. EXPRES  REGIONAL  TGV
0      0        0    1          0   0   0          0         0    0
1      0        0    1          0   0   0          0         0    1
2      0        0    0          0   0   0          1         0    0
3      0        1    0          0   0   0          0         0    0
4      0        0    0          1   0   0          0         0    0
5      1        0    0          0   0   0          0         0    0
6      0        0    0          0   1   1          0         0    0
7      0        0    0          0   0   0          0         1    0
8      0        0    1          0   0   1          0         0    0
9      0        0    1          0   1   0          0         0    0


In [17]:
ddf["origin"].str.replace('Ó', 'O').replace('Á', 'A').value_counts()

origin
MADRID    2000
Name: count, dtype: int64

In [16]:
ddf["destination"].str.replace('Ó', 'O').replace('Á', 'A').value_counts()

destination
SEVILLA      768
BARCELONA    706
VALENCIA     526
Name: count, dtype: int64

In [ ]:
# 1. ddf.drop(columns=["seats", "meta", "insert_date", "id", "company"])
# 2. drop na prices, fare, and vehicle_class
# 3. drop fares <= 50
# 4. multi-hot encode vehicle type
# 5. replace accentuation from origin and destination
# 6. convert departure/arrival to month, day, hour, minute columns

In [40]:
import pandas as pd
import dask.dataframe as dd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer


# Fast accent normalization via vectorized translate (much faster than per-row unicodedata)
ACCENT_TABLE = str.maketrans({
    "Á": "A", "À": "A", "É": "E", "È": "E",
    "Í": "I", "Ì": "I", "Ó": "O", "Ò": "O",
    "Ú": "U", "Ù": "U", "Ñ": "N",
    "á": "a", "à": "a", "é": "e", "è": "e",
    "í": "i", "ì": "i", "ó": "o", "ò": "o",
    "ú": "u", "ù": "u", "ñ": "n",
})


def _is_dask_df(df):
    return isinstance(df, dd.DataFrame)


def drop_unused_columns(df):
    cols_to_drop = ["seats", "meta", "insert_date", "id", "company"]
    existing = [c for c in cols_to_drop if c in df.columns]
    return df.drop(columns=existing)


def drop_missing_required(df):
    required = ["price", "fare", "vehicle_class"]
    existing = [c for c in required if c in df.columns]
    return df.dropna(subset=existing)


def keep_popular_fares(df, min_count=50, split_out=32):
    # Dask path: parallel value_counts reduction, then single filter pass.
    if _is_dask_df(df):
        fare_counts = df["fare"].value_counts(split_out=split_out).compute()
    else:
        fare_counts = df["fare"].value_counts()

    popular_fares = fare_counts[fare_counts > min_count].index
    return df[df["fare"].isin(popular_fares)]


def encode_vehicle_type(df):
    out = df

    vt = (
        out["vehicle_type"]
        .fillna("UNKNOWN")
        .astype(str)
        .str.upper()
        .str.strip()
        .replace({
            "R. EXPRES": "REG.EXP.",
            "INTERCITY": "INTERCITY",
        })
    )

    # Multi-hot in a vectorized way; avoids python-level loops.
    vehicle_multi_hot = vt.str.get_dummies(sep="-")
    vehicle_multi_hot = vehicle_multi_hot.add_prefix("vehicle_")

    out = out.drop(columns=["vehicle_type"], errors="ignore")
    out = out.join(vehicle_multi_hot)
    return out


def strip_accents_locations(df):
    out = df
    for col in ["origin", "destination"]:
        if col in out.columns:
            out[col] = out[col].astype(str).str.translate(ACCENT_TABLE)
    return out


def expand_datetime_parts(df):
    out = df

    for col in ["departure", "arrival"]:
        if col in out.columns:
            dt = dd.to_datetime(out[col], errors="coerce") if _is_dask_df(out) else pd.to_datetime(out[col], errors="coerce")

            # int8 with -1 sentinel keeps memory tiny and avoids nullable extension overhead.
            out[f"{col}_month"] = dt.dt.month.fillna(-1).astype("int8")
            out[f"{col}_day"] = dt.dt.day.fillna(-1).astype("int8")
            out[f"{col}_hour"] = dt.dt.hour.fillna(-1).astype("int8")
            out[f"{col}_minute"] = dt.dt.minute.fillna(-1).astype("int8")

    out = out.drop(columns=["departure", "arrival"], errors="ignore")
    return out


def encode_vehicle_class_binary(df):
    out = df
    if "vehicle_class" in out.columns:
        s = out["vehicle_class"].fillna("").astype(str).str.upper().str.strip()
        out["vehicle_class_bin"] = (~s.str.contains("PREFERENTE", regex=False)).astype("int8")
    return out


# Example for small pandas sample (your current ddf)
prepared_df = drop_unused_columns(ddf)
prepared_df = drop_missing_required(prepared_df)
prepared_df = keep_popular_fares(prepared_df, min_count=50)

post_pipeline = Pipeline([
    ("vehicle_multi_hot", FunctionTransformer(encode_vehicle_type, validate=False)),
    ("normalize_locations", FunctionTransformer(strip_accents_locations, validate=False)),
    ("datetime_features", FunctionTransformer(expand_datetime_parts, validate=False)),
    ("vehicle_class_binary", FunctionTransformer(encode_vehicle_class_binary, validate=False)),
])

model_df = post_pipeline.fit_transform(prepared_df)
model_df.drop(columns=["origin", "destination", "vehicle_class"], inplace=True)
model_df.head()

,duration,price,fare,vehicle_class_bin,vehicle_ALVIA,vehicle_AV CITY,vehicle_AVE,vehicle_INTERCITY,vehicle_LD,vehicle_MD,...,vehicle_REGIONAL,vehicle_TGV,departure_month,departure_day,departure_hour,departure_minute,arrival_month,arrival_day,arrival_hour,arrival_minute
0,3.08,68.95,Promo,0,0,0,1,0,0,0,...,0,0,4,18,5,50,4,18,8,55
1,2.98,107.70,Flexible,1,0,0,1,0,0,0,...,0,1,4,18,13,25,4,18,16,24
2,2.83,75.40,Promo,1,0,0,1,0,0,0,...,0,0,4,18,6,30,4,18,9,20
4,2.50,106.75,Promo,1,0,0,1,0,0,0,...,0,0,4,18,7,0,4,18,9,30
5,2.83,75.40,Promo,1,0,0,1,0,0,0,...,0,0,4,18,6,30,4,18,9,20


In [41]:
model_df.describe()

,duration,price,vehicle_class_bin,vehicle_ALVIA,vehicle_AV CITY,vehicle_AVE,vehicle_INTERCITY,vehicle_LD,vehicle_MD,vehicle_REG.EXP.,vehicle_REGIONAL,vehicle_TGV,departure_month,departure_day,departure_hour,departure_minute,arrival_month,arrival_day,arrival_hour,arrival_minute
count,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000,1758.000000
mean,2.850262,64.994420,0.895336,0.019340,0.038680,0.837315,0.039818,0.011377,0.004551,0.012514,0.047782,0.017065,4.763936,16.146758,13.565415,19.656428,4.763936,16.167235,15.746303,30.344141
std,1.276030,26.273096,0.306208,0.137757,0.192887,0.369183,0.195587,0.106083,0.067324,0.111196,0.213364,0.129550,0.571089,9.311780,4.614568,16.810106,0.571089,9.304586,4.995778,13.111813
min,1.670000,15.450000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,1.000000,5.000000,0.000000,4.000000,1.000000,0.000000,3.000000
25%,2.500000,46.150000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,10.000000,9.000000,0.000000,4.000000,10.000000,12.000000,21.000000
50%,2.580000,66.550000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,17.000000,14.000000,25.000000,5.000000,17.000000,16.000000,30.000000
75%,2.930000,76.300000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,22.000000,17.000000,30.000000,5.000000,22.000000,20.000000,40.000000
max,9.370000,214.200000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,6.000000,30.000000,21.000000,55.000000,6.000000,30.000000,23.000000,58.000000
